## Load User Data

In [1]:
BANNED_USERS = {51, 86, 62, 39, 66, 89, 67, 57} # failed attention checks

import json
with open('../data/user_code.json') as json_data:
    d = json.load(json_data)
    user_to_scores = {
        row['user_id']: {
            'background': float(row['background_ability']),
            'comprehension': float(row['authorship_and_comprehension'])
        } for row in d if row['user_id'] not in BANNED_USERS
    }
    user_id_to_group = {
        row['user_id']: row['experiment_group'] for row in d if row['user_id'] not in BANNED_USERS
    }

## Prompt Statistics

In [2]:
import json
import pandas as pd

user_to_num_prompts = dict()
with open('../data/ai_trace.json') as json_data:
    data = json.load(json_data)
    for row in data:
        if row['user_id'] in BANNED_USERS:
            continue
        key = (row['user_id'], row['project_id'])
        user_to_num_prompts[key] = user_to_num_prompts.get(key, 0) + 1
        
prompt_dataset = {
    'user': [],
    'key': [],
    'num': [],
}
for (user, task), v in user_to_num_prompts.items():
    prompt_dataset['user'].append(user)
    prompt_dataset['key'].append((user_id_to_group[user], task))
    prompt_dataset['num'].append(v)
prompt_dataset = pd.DataFrame(prompt_dataset)

from scipy.stats import sem
print("Prompt Usage:")
prompt_dataset.groupby("key")["num"].agg(mean="mean", stderr=sem).reset_index()

Prompt Usage:


,key,mean,stderr
0,"(agent, extension)",6.782609,0.882318
1,"(agent, initial)",5.769231,0.441353
2,"(chat, extension)",4.736842,0.576550
3,"(chat, initial)",11.571429,2.505776


## Analyze Agent Prompts

In [3]:
import json

with open('../llm_judge/results/cluster/label_agent_prompts.jsonl', 'r') as json_file:
    json_list = list(json_file)

label_to_users = dict()
    
for json_str in json_list:
    result = json.loads(json_str)
    if result['user_id'] in BANNED_USERS:
        continue
    label = result['label']
    arr = label_to_users.get(label, [])
    arr.append(result['user_id'])
    label_to_users[label] = arr
    
# we merged the 'copy' and 'rephrased' labels, due to low human agreement and realizing that they test the same prompt interaction type
label_to_users['copy'] = label_to_users['copy'] + label_to_users['rephrased']
del label_to_users['rephrased']
label_to_users_set = {k: set(v) for k, v in label_to_users.items()}
total_num = sum([len(v) for v in label_to_users.values()])

In [4]:
import numpy as np
print("## Table 1\n======================")
for label, users in label_to_users_set.items():
    user_bg = [user_to_scores[user]['background'] for user in users]
    user_comp = [user_to_scores[user]['comprehension'] for user in users]
    
    print("Label:", label)
    print("Comp:", np.mean(user_comp))
    print("BG:", np.mean(user_bg))
    print("# Users:", len(users))
    print("Proportion:", (1.0 * len(label_to_users[label])) / total_num)
    print('----------------------')

## Table 1
Label: technical
Comp: 0.7428451178451181
BG: 0.6837606837606837
# Users: 9
Proportion: 0.08666666666666667
----------------------
Label: debugging
Comp: 0.6916666666666668
BG: 0.6615384615384615
# Users: 5
Proportion: 0.04
----------------------
Label: copy
Comp: 0.6540909090909092
BG: 0.6676923076923077
# Users: 25
Proportion: 0.7933333333333333
----------------------
Label: exploratory
Comp: 0.6351010101010102
BG: 0.6923076923076922
# Users: 6
Proportion: 0.04666666666666667
----------------------
Label: other
Comp: 0.7666666666666668
BG: 0.8
# Users: 5
Proportion: 0.03333333333333333
----------------------


In [26]:
# Appendix A.4: Regression of interaction types

import json
import numpy as np

with open('../llm_judge/results/cluster/label_agent_prompts.jsonl', 'r') as json_file:
    json_list = list(json_file)

# Get the prompt types from each user
user_to_agent_prompt = dict()    
for json_str in json_list:
    result = json.loads(json_str)
    if result['user_id'] in BANNED_USERS or result['label'] == 'other':
        continue
    arr = user_to_agent_prompt.get(result['user_id'], [])
    arr.append(result['label'].replace('rephrased', 'copy'))
    user_to_agent_prompt[result['user_id']] = arr

# Parse dataframe
keys = ['copy', 'technical', 'debugging', 'exploratory']
prompt_df = {
    k: [] for k in keys + ['user_id', 'comprehension', 'background']
}
for k, arr in user_to_agent_prompt.items():
    prompt_df['user_id'].append(k)
    prompt_df['comprehension'].append(user_to_scores[k]['comprehension'])
    prompt_df['background'].append(user_to_scores[k]['background'])
    for k in keys:
        prompt_df[k].append(np.mean([elem == k for elem in arr]))
prompt_df = pd.DataFrame(prompt_df)

# Regression: use the proportion of each prompt type to predict comprehension, controlling for BG ability
import statsmodels.formula.api as smf
formula = f"comprehension ~ background + copy + exploratory + debugging"
res = smf.ols(formula, data=prompt_df).fit()
res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:          comprehension   R-squared:                       0.563
Model:                            OLS   Adj. R-squared:                  0.476
Method:                 Least Squares   F-statistic:                     6.445
Date:                Tue, 05 May 2026   Prob (F-statistic):            0.00168
Time:                        12:11:14   Log-Likelihood:                 21.366
No. Observations:                  25   AIC:                            -32.73
Df Residuals:                      20   BIC:                            -26.64
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       0.7141      0.225      3.176      0.005       0.245       1.183
background      0.5688      0.128      4.440      0.000       0.302       0.836
copy           -0.4685      0.217     -2.161      0.043      -0.921      -0.016
exploratory    -0.7447      0.387     -1.923      0.069      -1.553       0.063
debugging      -0.3378      0.361     -0.936      0.361      -1.091       0.415
==============================================================================
Omnibus:                        4.106   Durbin-Watson:                   2.015
Prob(Omnibus):                  0.128   Jarque-Bera (JB):                2.329
Skew:                          -0.554   Prob(JB):                        0.312
Kurtosis:                       4.004   Cond. No.                         35.0
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

## Analyze Chatbot Prompts

In [6]:
import json

with open('../llm_judge/results/cluster/label_chatbot_prompts.jsonl', 'r') as json_file:
    json_list = list(json_file)

label_to_users = dict()
    
for json_str in json_list:
    result = json.loads(json_str)
    if result['user_id'] in BANNED_USERS:
        continue
    label = result['label']
    arr = label_to_users.get(label, [])
    arr.append(result['user_id'])
    label_to_users[label] = arr
    
label_to_users_set = {k: set(v) for k, v in label_to_users.items()}
total_num = sum([len(v) for v in label_to_users.values()])

In [7]:
import numpy as np
print("## Appendix Table 4\n======================")
for label, users in label_to_users_set.items():
    user_bg = [user_to_scores[user]['background'] for user in users]
    user_comp = [user_to_scores[user]['comprehension'] for user in users]
    
    print("Label:", label)
    print("Comp:", np.mean(user_comp))
    print("BG:", np.mean(user_bg))
    print("# Users:", len(users))
    print("Proportion:", (1.0 * len(label_to_users[label])) / total_num)
    print('----------------------')

## Appendix Table 4
Label: syntax_help
Comp: 0.8255561568061568
BG: 0.6446886446886448
# Users: 21
Proportion: 0.4279835390946502
----------------------
Label: design_help
Comp: 0.8335182178932179
BG: 0.6263736263736265
# Users: 14
Proportion: 0.1934156378600823
----------------------
Label: clarification
Comp: 0.8112201561065198
BG: 0.5804195804195804
# Users: 11
Proportion: 0.16872427983539096
----------------------
Label: snippet
Comp: 0.8527076318742987
BG: 0.658119658119658
# Users: 9
Proportion: 0.09465020576131687
----------------------
Label: debugging
Comp: 0.7805555555555554
BG: 0.49230769230769234
# Users: 5
Proportion: 0.05761316872427984
----------------------
Label: jailbreaking
Comp: 0.7545454545454546
BG: 0.5692307692307692
# Users: 5
Proportion: 0.037037037037037035
----------------------
Label: urgency
Comp: 0.583333333333333
BG: 0.46153846153846156
# Users: 1
Proportion: 0.0205761316872428
----------------------


## AI Review Types

In [8]:
import json
with open('../data/ai_approval.json') as json_data:
    approval_data = json.load(json_data)

In [9]:
# build a user history of actions
user_history = dict()
approval_data.sort(key=lambda item: item['created_at'])
for row in approval_data:
    user_id = row['user_id']
    if user_id in BANNED_USERS:
        continue
    curr_history = user_history.get(user_id, [])
    curr_history.append(row['mode'])
    user_history[user_id] = curr_history

In [10]:
def classify_span(curr_span):
    label = ''
    if 'diff' in curr_span or 'reject' in curr_span or 'reject_all' in curr_span:
        label = 'manual_edit'
    elif 'keep_all' in curr_span:
        label = 'click_accept_all'
    elif 'keep' in curr_span or 'keep_all' in curr_span:
        label = 'click_accept_each'
    else:
        label = 'auto_accept'
    return label

review_label_to_users = {
    'auto_accept': [],
    'manual_edit': [],
    'click_accept_each': [],
    'click_accept_all': [],
}

for user, hist in user_history.items():
    ai_idxs = [idx for idx, elem in enumerate(hist) if elem == 'AI']
    if not ai_idxs:
        continue
        
    start_idx = ai_idxs.pop(0)
    while ai_idxs:
        end_idx = ai_idxs.pop(0)
        span = hist[start_idx:end_idx]
        review_label_to_users[classify_span(span)].append(user)
        start_idx = end_idx
        
    span = hist[start_idx:]
    review_label_to_users[classify_span(span)].append(user)
    
review_label_to_users_set = {k: set(v) for k, v in review_label_to_users.items()}
total_num = sum([len(v) for v in review_label_to_users.values()])

In [11]:
import numpy as np
print("## Table 2\n======================")
for label, users in review_label_to_users_set.items():
    user_bg = [user_to_scores[user]['background'] for user in users]
    user_comp = [user_to_scores[user]['comprehension'] for user in users]
    
    print("Label:", label)
    print("Comp:", np.mean(user_comp))
    print("BG:", np.mean(user_bg))
    print("# Users:", len(users))
    print("Proportion:", (1.0 * len(review_label_to_users[label])) / total_num)
    print('----------------------')

## Table 2
Label: auto_accept
Comp: 0.6145833333333335
BG: 0.6025641025641025
# Users: 6
Proportion: 0.05303030303030303
----------------------
Label: manual_edit
Comp: 0.6609848484848484
BG: 0.6820512820512821
# Users: 15
Proportion: 0.3181818181818182
----------------------
Label: click_accept_each
Comp: 0.7765151515151516
BG: 0.7142857142857143
# Users: 7
Proportion: 0.13636363636363635
----------------------
Label: click_accept_all
Comp: 0.6644886363636363
BG: 0.6692307692307693
# Users: 20
Proportion: 0.49242424242424243
----------------------


In [35]:
# Appendix A.4: Regression of code review types

import json
import numpy as np

with open('../llm_judge/results/cluster/label_agent_prompts.jsonl', 'r') as json_file:
    json_list = list(json_file)

# Get the prompt types from each user
user_to_review = dict()    

for user, hist in user_history.items():
    ai_idxs = [idx for idx, elem in enumerate(hist) if elem == 'AI']
    if not ai_idxs:
        continue
        
    start_idx = ai_idxs.pop(0)
    while ai_idxs:
        end_idx = ai_idxs.pop(0)
        span = hist[start_idx:end_idx]
        review_label_to_users[classify_span(span)].append(user)
        start_idx = end_idx
        if user not in user_to_review:
            user_to_review[user] = []
        user_to_review[user].append(classify_span(span))
        
    span = hist[start_idx:]
    if user not in user_to_review:
        user_to_review[user] = []
    user_to_review[user].append(classify_span(span))

# Parse dataframe
keys = ['click_accept_all', 'click_accept_each', 'manual_edit', 'auto_accept']
review_df = {
    k: [] for k in keys + ['user_id', 'comprehension', 'background']
}
for k, arr in user_to_review.items():
    review_df['user_id'].append(k)
    review_df['comprehension'].append(user_to_scores[k]['comprehension'])
    review_df['background'].append(user_to_scores[k]['background'])
    for k in keys:
        review_df[k].append(np.mean([elem == k for elem in arr]))
review_df = pd.DataFrame(review_df)

# Regression: use the proportion of each review type to predict comprehension, controlling for BG ability
import statsmodels.formula.api as smf
formula = f"comprehension ~ background + click_accept_all + manual_edit + auto_accept"
res = smf.ols(formula, data=review_df).fit()
res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:          comprehension   R-squared:                       0.563
Model:                            OLS   Adj. R-squared:                  0.480
Method:                 Least Squares   F-statistic:                     6.759
Date:                Tue, 05 May 2026   Prob (F-statistic):            0.00116
Time:                        12:43:21   Log-Likelihood:                 22.338
No. Observations:                  26   AIC:                            -34.68
Df Residuals:                      21   BIC:                            -28.39
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
====================================================================================
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            0.4751      0.131      3.633      0.002       0.203       0.747
background           0.5088      0.127      4.011      0.001       0.245       0.773
click_accept_all    -0.1585      0.093     -1.709      0.102      -0.351       0.034
manual_edit         -0.2136      0.100     -2.146      0.044      -0.421      -0.007
auto_accept         -0.3343      0.261     -1.281      0.214      -0.877       0.208
==============================================================================
Omnibus:                        7.152   Durbin-Watson:                   2.296
Prob(Omnibus):                  0.028   Jarque-Bera (JB):                5.313
Skew:                          -0.813   Prob(JB):                       0.0702
Kurtosis:                       4.505   Cond. No.                         17.0
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""